<a href="https://colab.research.google.com/github/alimovscott/cloneGPT/blob/main/clone_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers torch bitsandbytes datasets peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 9.9 MB/s eta 0:00:00


In [19]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, TrainingArguments
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

In [4]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
tokenizer = AutoTokenizer.from_pretrained(model_id)

# print("Vocab size:", tokenizer.vocab_size)
# print('Special tokens:', tokenizer.special_tokens_map)


# quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    # dtype=torch.bfloat16

    )
#

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
# Before Fine-tuning
promt = "Explain what a tokenizer is? "
# promt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(
    promt,
    return_tensors='pt'
).to(model.device)

with torch.no_grad():
  output_ids = model.generate(
      **inputs,
      max_new_tokens=80,
      do_sample=True,
      temperature=0.7
  )

  print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is? 2.2 Explain the roles of the lexicon and the parser in the generation of texts? 2.3 Analyze the effectiveness of different tokenization methods, including Whitespace Tokenization, Word Tokenization, and Part-of-speech (POS) Tokenization, in generating text data from different sources. 2.4 Explain the different approaches to part-


In [6]:
def count_parameters(model):
  return sum(p.numel() for p in model.parameters() )

total_params = count_parameters(model)
print(f'Total parameters: {total_params:,}')



Total parameters: 615,606,272


In [7]:
## datasets
## instraction tuning

In [20]:

dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset[0]

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'input': '',
 'instruction': 'Give three tips for staying healthy.'}

In [9]:
def generate_prompt(example):
  instruction = example['instruction']
  input_text = example['input']
  output_text = example['output']

  if input_text:
    return(
    "### Instruction:\n"
    f"{instruction}\n\n"
    "### Input:\n\n"
    f"{input_text}\n\n"
    "### Response:\n"
    f"{output_text}"
    )
  else:
    return(
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Response:\n"
        f"{output_text}"
    )


# generate_prompt(dataset[1])


def formatting_func(example):
  return{'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)




Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [10]:
dataset[0]['text']

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'

In [11]:
dataset = dataset.select(range(7000))

In [12]:
dataset = dataset.shuffle(seed=42)

In [13]:
# Full Fine- tuning =>
# cheap Fine-tuning =>
# PEFT => paremetr Efficent Fine Tuning
# OOM => Out of Memory

In [21]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)




In [15]:
model = get_peft_model(model, lora_config)


In [16]:
model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [ ]:
# QLora
# Lora